# Stage 9 — Debit spread live screener

New: `src/debit_spread_screener.py` — parallel to `daily_screener.py`, not
a modification of it. That one checks a stock's currently-open Supertrend
trend AGE; this checks for a currently-OPEN debit-spread TRADE (a signal
that fired recently and hasn't yet resolved) - the actionable entry
itself, with live stop/target and candles left before the time-exit
forces a close.

Same two-speed discipline as the existing screener: deep full-universe
refresh (notebook 08, ~monthly) keeps the ranking honest; THIS notebook
does a quick, short-lookback refresh of only the shortlist, right before
checking - never point this at the full universe.

**Regression fixed:** this refresh cell used to call `process_instrument()`, which does a full replace of the raw 1H file - every run was silently overwriting notebook 08's deep 2023-2026 history for whatever's in the shortlist with just a 120-day window (same class of bug found in the pledged-securities and option-buying pipelines - discovered here because this notebook's own earlier run is what first shrank ADANIENSOL/KPITTECH/BSE's shared 1H file, which then propagated into the option-buying pipeline too since both strategies read/write the same raw files). Now uses `process_instrument_incremental()`, which merges the quick catch-up window onto existing history instead of replacing it, and refuses to save if the merge would shrink history. If you ran this notebook before the fix, re-run notebook 08 (full-universe deep refresh) once to restore any shortlisted symbols that got shrunk.

In [1]:
import sys
sys.path.append("..")

import pandas as pd

ranking = pd.read_excel("../exports/RANKING_debit_spread.xlsx")

# Keep only outlier-reliance-clean stocks, take the top 20
clean_candidates = ranking[
    ranking["outlier_reliance_flag"].isna() | (ranking["outlier_reliance_flag"] == "")
]
top_20 = clean_candidates.head(20)

# Always include NIFTY and BANKNIFTY, same flag check
must_include = ranking[ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["outlier_reliance_flag"].isna() | (must_include["outlier_reliance_flag"] == "")]

final_candidates = pd.concat([top_20, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates[["symbol", "debit_spread_score", "win_rate_pct", "avg_r_achieved", "median_r_achieved"]]

,symbol,debit_spread_score,win_rate_pct,avg_r_achieved,median_r_achieved
0,POWERINDIA,98.9,64.2,0.54,1.50
1,TIINDIA,95.0,54.5,0.28,0.61
2,CDSL,94.5,57.4,0.41,1.46
3,INFY,93.5,56.0,0.34,0.82
4,GAIL,92.7,52.4,0.27,0.50
5,PFC,92.5,54.1,0.36,0.95
6,SUZLON,92.1,54.9,0.33,0.64
7,ADANIENSOL,91.9,52.9,0.29,0.28
8,ANGELONE,91.8,53.9,0.33,0.74
9,BHEL,90.9,55.9,0.30,0.34


In [2]:
from datetime import datetime
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import process_instrument_incremental

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

# REGRESSION FIX: this used to call process_instrument() with a short
# 120-day from_date, which does a full REPLACE of the raw 1H file - that
# silently destroyed notebook 08's deep 2023-2026 history for every
# shortlisted symbol on every screener run (confirmed: this notebook's own
# earlier run already shrank ADANIENSOL/KPITTECH/BSE's shared 1H file to
# 572 candles). Now uses process_instrument_incremental() instead - same
# quick 120-day catch-up fetch, but MERGED onto existing saved history
# rather than replacing it, with a guard that refuses to save if history
# would shrink. No from_date needed here; the incremental function derives
# its own fetch window from what's already saved.

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_instrument_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-04-09 to 2026-08-07...
Saved 6195 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/POWERINDIA_60minute.csv
Saved 67 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/POWERINDIA_trades.csv
[1/20] POWERINDIA — refreshed
Fetching 2026-04-09 to 2026-08-07...
Saved 6195 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/TIINDIA_60minute.csv
Saved 66 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/TIINDIA_trades.csv
[2/20] TIINDIA — refreshed
Fetching 2026-04-09 to 2026-08-07...
Saved 6195 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/CDSL_60minute.csv
Saved 68 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal

In [3]:
from src.debit_spread_screener import screen_shortlist

active = screen_shortlist(symbols_to_refresh)
print(f"{len(active)} of {len(symbols_to_refresh)} shortlisted instruments have an active trade right now")
active

2 of 20 shortlisted instruments have an active trade right now


,symbol,direction,entry_date,entry_price,stop_price,target_price,candles_held,candles_left_before_time_exit,current_pct_move,current_r
0,HAL,UP,2026-08-06 09:15:00,4877.2,4809.68,4978.48,7,7,0.45,0.33
1,KALYANKJIL,UP,2026-08-07 10:15:00,622.5,605.25,648.38,0,14,0.00,0.00
